In [4]:

from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
import numpy as np
from sklearn.linear_model import SGDRegressor
X,y = load_diabetes(return_X_y=True)
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=4)
reg = SGDRegressor(penalty='l2',max_iter=5000,eta0=0.1,learning_rate='constant',alpha=0.001)
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.45952683980680087
[  50.65652646 -152.55855426  368.64589436  267.24109866   -4.70413621
  -57.93261705 -168.90007037  138.41342529  330.06041743   96.84406643]
[158.83054361]


In [5]:
from sklearn.linear_model import Ridge

reg = Ridge(alpha=0.001, max_iter=5000,solver='sparse_cg')
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.4625010162027918
[  34.52192778 -290.84083871  482.40181675  368.06786931 -852.44872818
  501.59160694  180.11115474  270.76334443  759.73534802   37.49135796]
151.101985182554


In [6]:
## Sgd

In [7]:
class MeraRidgeGD:
    
    def __init__(self,epochs,learning_rate,alpha):
        
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self,X_train,y_train):
        
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        thetha = np.insert(self.coef_,0,self.intercept_)
        
        X_train = np.insert(X_train,0,1,axis=1)
        
        for i in range(self.epochs):
            thetha_der = np.dot(X_train.T,X_train).dot(thetha) - np.dot(X_train.T,y_train) + self.alpha*thetha
            thetha = thetha - self.learning_rate*thetha_der
        
        self.coef_ = thetha[1:]
        self.intercept_ = thetha[0]
    
    def predict(self,X_test):
        
        return np.dot(X_test,self.coef_) + self.intercept_

In [9]:
reg = MeraRidgeGD(epochs=5000,alpha=0.01,learning_rate=0.005)
reg.fit(X_train,y_train)

y_pred = reg.predict(X_test)
print("R2 score",r2_score(y_test,y_pred))
print(reg.coef_)
print(reg.intercept_)

R2 score 0.4678913228577495
[  41.3489941  -280.9481324   489.10217578  361.0264397  -160.57497191
  -32.97881557 -138.4889039   163.83670988  497.50554332   36.43718559]
150.82937664180704


In [10]:
import numpy as np

class MeraRidgeGD:

    def __init__(self, epochs=1000, learning_rate=0.01, alpha=1.0, batch_size=32):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.batch_size = batch_size
        
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):

        n_samples, n_features = X_train.shape
        
        # add bias column
        X_train = np.insert(X_train, 0, 1, axis=1)

        theta = np.zeros(n_features + 1)

        for epoch in range(self.epochs):

            # shuffle every epoch (IMPORTANT)
            indices = np.random.permutation(n_samples)
            X_train = X_train[indices]
            y_train = y_train[indices]

            # mini-batches
            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size

                X_batch = X_train[start:end]
                y_batch = y_train[start:end]

                predictions = X_batch @ theta
                errors = predictions - y_batch

                gradient = (X_batch.T @ errors) / len(X_batch)

                # Ridge penalty (skip intercept)
                gradient[1:] += self.alpha * theta[1:]

                theta -= self.learning_rate * gradient

        self.intercept_ = theta[0]
        self.coef_ = theta[1:]

    def predict(self, X_test):
        return X_test @ self.coef_ + self.intercept_


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MeraRidgeGD(epochs=1500, learning_rate=0.001, alpha=1, batch_size=32)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(r2_score(y_test,y_pred))

0.4171202384126902
